In [8]:
import sys
sys.path.insert(0, ".")

import json
from pathlib import Path
from tqdm.notebook import tqdm
from src.captioner import Captioner

In [ ]:
# # Run this if frame captures are regenerated 
# import os
# os.remove("output/descriptions.jsonl")
# print("Cleared old descriptions ✅")

Cleared old descriptions ✅


In [9]:
captioner = Captioner()

Loading BLIP on mps...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Captioner ready ✅


In [ ]:
# run on full manifest
MANIFEST_PATH     = Path("output/manifest.jsonl")
DESCRIPTIONS_PATH = Path("output/descriptions.jsonl")

with open(MANIFEST_PATH) as f_in, \
     open(DESCRIPTIONS_PATH, "w") as f_out:

    for line in tqdm(f_in):
        entry    = json.loads(line)
        video_id = entry["video_id"]
        enriched_segments = []

        for seg in entry["segments"]:
            frame_descriptions = []
            for frame_path in seg["frame_paths"]:
                desc = captioner.caption_frame(frame_path)
                frame_descriptions.append(desc)

            enriched_segments.append({
                **seg,
                "frame_descriptions": frame_descriptions,
            })

        f_out.write(json.dumps({
            **entry,
            "segments": enriched_segments,
        }) + "\n")

print("Done ✅")

0it [00:00, ?it/s]

In [4]:
with open(DESCRIPTIONS_PATH) as f:
    entry = json.loads(f.readline())

print(f"Video    : {entry['video_id']}")
print(f"Caption  : {entry['caption']}")
print()
for seg in entry["segments"]:
    print(f"Segment {seg['segment_index']} [{seg['timestamp_start']:.1f}s → {seg['timestamp_end']:.1f}s]")
    print(f"  Ground truth : {seg['sentence']}")
    for ft, desc in zip(seg["frame_timestamps"], seg["frame_descriptions"]):
        print(f"  {ft:.1f}s : {desc}")
    print()

Video    : v_QOlSCBRmfWY
Caption  : A young woman is seen standing in a room and leads into her dancing. The girl dances around the room while the camera captures her movements. She continues dancing around the room and ends by laying on the floor.

Segment 0 [0.8s → 19.9s]
  Ground truth : A young woman is seen standing in a room and leads into her dancing.
  0.8s : a woman in a black top is standing on the floor
  7.2s : a woman is dancing in an empty room
  13.5s : a woman standing on one leg in an empty room
  19.9s : a woman is laying on the floor in an empty room

Segment 1 [17.4s → 60.8s]
  Ground truth : The girl dances around the room while the camera captures her movements.
  17.4s : a woman doing a handstant on the floor
  23.6s : a woman is doing a handstant on the floor
  29.8s : a woman is sitting on the floor in an empty room
  36.0s : a woman is dancing in an empty room
  42.2s : a person doing a handstant in a room
  48.4s : a woman laying on the floor in a gym
  54.6s

In [5]:
import csv
import json
from pathlib import Path

DESCRIPTIONS_PATH = Path("output/descriptions.jsonl")
TRAINING_PATH     = Path("output/training_data.csv")

rows = []

with open(DESCRIPTIONS_PATH) as f:
    for line in f:
        entry    = json.loads(line)
        video_id = entry["video_id"]
        caption  = entry["caption"]          # full video caption
        segments = entry["segments"]
        n_segs   = len(segments)

        for seg in segments:
            # Build input string: timestamped frame descriptions
            frame_lines = []
            for ft, desc in zip(seg["frame_timestamps"], seg["frame_descriptions"]):
                frame_lines.append(f"[{ft:.1f}s] {desc}")
            input_text = " | ".join(frame_lines)

            rows.append({
                "video_id"       : video_id,
                "segment_index"  : seg["segment_index"],
                "n_segments"     : n_segs,
                "timestamp_start": seg["timestamp_start"],
                "timestamp_end"  : seg["timestamp_end"],
                "input"          : input_text,         # ← frame descriptions chained
                "target_sentence": seg["sentence"],    # ← ground truth for this segment
                "target_caption" : caption,            # ← ground truth for full video
            })

with open(TRAINING_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {TRAINING_PATH}")

Saved 34 rows to output/training_data.csv


In [6]:
import pandas as pd

df = pd.read_csv(TRAINING_PATH)
print(df.shape)
df.head(3)

(34, 8)


,video_id,segment_index,n_segments,timestamp_start,timestamp_end,input,target_sentence,target_caption
0,v_QOlSCBRmfWY,0,3,0.83,19.86,[0.8s] a woman in a black top is standing on t...,A young woman is seen standing in a room and l...,A young woman is seen standing in a room and l...
1,v_QOlSCBRmfWY,1,3,17.37,60.81,[17.4s] a woman doing a handstant on the floor...,The girl dances around the room while the came...,A young woman is seen standing in a room and l...
2,v_QOlSCBRmfWY,2,3,56.26,79.42,[56.3s] a woman is doing a handstant on the fl...,She continues dancing around the room and ends...,A young woman is seen standing in a room and l...


In [ ]:
# Run in any notebook first
ds_full = load_dataset("friedrichor/ActivityNet_Captions", split="train", streaming=True)
count = sum(1 for _ in ds_full)
print(f"Total videos: {count}")